# This is a simple example script for 2D x/y stitching using muvis-align and the multiview-stitcher package

In [ ]:
import sys
sys.path.append('..')

from multiview_stitcher import spatial_image_utils as si_utils, param_utils
from multiview_stitcher import vis_utils
import networkx as nx
import numpy as np

from muvis_align.MVSRegistration import MVSRegistration
from muvis_align.image.util import get_sim_physical_size, show_image, extract_sims_from_fused, \
    sims_from_sims_or_msims

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_path='../data/S000/*.zarr', output_path='../../output/', ui='mpl')
reg.init_data()
msims = reg.msims
register_msims, reg_indices, _ = reg.preprocess(msims)
sims = sims_from_sims_or_msims(msims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Initialise registration parameters

In [ ]:
register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'rigid',
	'method': 'sift',
	'gaussian_sigma': 2,
	'normalisation': True,
	'max_keypoints': 5000,
	'inlier_threshold_factor': 0.05,
	'max_trials': 1000,
	'ransac_iterations': 3,
	'n_parallel_pairwise_regs': 1,
}

## Perform pair registration (using multiview-stitcher)

In [ ]:
reg_results = reg.register_pairs(register_msims=register_msims, params=register_params)

## Show pairwise registration results

In [ ]:
pairs_graph = reg_results['pairs_graph']
transforms = nx.get_edge_attributes(pairs_graph, 'transform')
qualities = nx.get_edge_attributes(pairs_graph, 'quality')
for key in transforms:
    print(f'{key} quality={float(qualities[key].item()):.3} transform:\n{np.array(transforms[key])}')

## Pair transforms modification

In [ ]:
transforms[0, 1] = param_utils.identity_transform(2)    # modify first transform to eye transform
qualities[0, 1] = np.array(1)    # set quality to 1
nx.set_edge_attributes(pairs_graph, transforms, 'transform')
nx.set_edge_attributes(pairs_graph, qualities, 'quality')

## Perfrom global registration (using multiview-stitcher)

In [ ]:
%matplotlib inline
pair_msims = reg_results['msims']
results = reg.register_global(pair_msims, register_indices=reg_indices, params=register_params)

## Show registration mapping

In [ ]:
mappings = results['mappings']
for key, mapping in mappings.items():
    print(f'{reg.file_labels[key]}:\n', mapping.sel(t=0).data)

## Visualise registered sims

In [ ]:
%matplotlib inline
sims = sims_from_sims_or_msims(msims)
fig, ax = vis_utils.plot_positions(sims, transform_key=reg.reg_transform_key, use_positional_colors=False, view_labels=reg.file_labels)

## Perform fusion (using multiview-stitcher)

In [ ]:
%matplotlib inline
fused_msim, _ = reg.fuse(msims)

fused_msim

## Output fused result

In [ ]:
%matplotlib inline
fused_sim = extract_sims_from_fused(fused_msim)
show_image(fused_sim[0, 0])

In [ ]:
reg.save('stitching2d', fused_sim)
print('Done')